# 05 · Guidance & Conditioning

> **Source notes:** `GuidanceConditioning.md`

How do you make a diffusion model follow a class label or text prompt?

This notebook demonstrates:
- **Class embedding** — inject digit label into the U-Net
- **Condition dropout** — train with random null conditioning (CFG setup)
- **CFG sampling** — run two passes per step, amplify toward condition
- **Guidance scale sweep** — show how different $w$ values affect output
- **PixelSmith v3.5** — conditionally generate specific digit classes

**All local. No GPU needed.** Runtime: ~4 minutes (training included).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy", "-q"], check=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

print("Packages ready.")

In [ ]:
# Noise schedule (same as Ch.4)
T_STEPS = 1000
betas     = torch.linspace(1e-4, 0.02, T_STEPS)
alphas    = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)
sqrt_ab   = alpha_bar.sqrt()
sqrt_1mab = (1 - alpha_bar).sqrt()

def q_sample(x0, t):
    eps  = torch.randn_like(x0)
    xt   = sqrt_ab[t].view(-1,1,1,1) * x0 + sqrt_1mab[t].view(-1,1,1,1) * eps
    return xt, eps

## 1 · Class-Conditional U-Net

Add a learnable class embedding to the timestep embedding. One extra token: the digit class (0–9) or a null token (10) for unconditional generation.

In [ ]:
NUM_CLASSES = 10   # digits 0–9
NULL_CLASS  = 10   # special null token for unconditional training

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-torch.arange(half, device=t.device) * (np.log(10000) / (half-1)))
        args  = t[:,None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(time_dim, out_ch)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = F.silu(self.conv1(x))
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = F.silu(self.conv2(h))
        return h + self.skip(x)

class CondUNet(nn.Module):
    """Class-conditional U-Net for MNIST with CFG support."""
    def __init__(self, base_ch=32, time_dim=64):
        super().__init__()
        # NULL_CLASS+1 = 11 embeddings: classes 0-9 + null token
        self.class_embed = nn.Embedding(NUM_CLASSES + 1, time_dim)
        self.time_embed  = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )
        C = base_ch
        self.enc1 = ResBlock(1,   C,   time_dim)
        self.enc2 = ResBlock(C,   C*2, time_dim)
        self.enc3 = ResBlock(C*2, C*4, time_dim)
        self.bot  = ResBlock(C*4, C*4, time_dim)
        self.dec3 = ResBlock(C*8, C*2, time_dim)
        self.dec2 = ResBlock(C*4, C,   time_dim)
        self.dec1 = ResBlock(C*2, C,   time_dim)
        self.out  = nn.Conv2d(C, 1, 1)
        self.down = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode="nearest")

    def forward(self, x, t, c):
        # Combine time + class embeddings
        t_emb = self.time_embed(t) + self.class_embed(c)  # (B, time_dim)
        s1 = self.enc1(x, t_emb)
        s2 = self.enc2(self.down(s1), t_emb)
        s3 = self.enc3(self.down(s2), t_emb)
        b  = self.bot(s3, t_emb)
        d3 = self.dec3(torch.cat([self.up(b), s3], 1), t_emb)
        d2 = self.dec2(torch.cat([self.up(d3), s2], 1), t_emb)
        d1 = self.dec1(torch.cat([d2, s1], 1), t_emb)
        return self.out(d1)

model = CondUNet(base_ch=32, time_dim=64)
print(f"CondUNet parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Extra params vs unconditional: {sum(p.numel() for p in model.class_embed.parameters()):,} "
      f"(class embedding table: {NUM_CLASSES+1} × 64)")

## 2 · CFG Training — Condition Dropout

10% of the time, replace the true class label with the null token `NULL_CLASS=10`. This trains the model to generate with OR without conditioning.

In [ ]:
EPOCHS     = 25
BATCH_SIZE = 256
P_UNCOND   = 0.10   # condition dropout probability

dataset = torchvision.datasets.MNIST(
    root="/tmp/mnist", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Lambda(lambda x: x*2-1)])
)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
optim   = torch.optim.Adam(model.parameters(), lr=2e-4)

losses = []
model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for x0, c in loader:
        t     = torch.randint(0, T_STEPS, (x0.shape[0],))
        x_t, eps = q_sample(x0, t)
        
        # Condition dropout: replace label with null token with prob P_UNCOND
        mask  = (torch.rand(x0.shape[0]) < P_UNCOND)
        c_in  = c.clone()
        c_in[mask] = NULL_CLASS
        
        eps_pred = model(x_t, t, c_in)
        loss     = F.mse_loss(eps_pred, eps)
        optim.zero_grad(); loss.backward(); optim.step()
        epoch_loss += loss.item()
    
    avg = epoch_loss / len(loader)
    losses.append(avg)
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  loss={avg:.4f}")

print("Training complete.")

## 3 · CFG Sampling

In [ ]:
@torch.no_grad()
def cfg_sample(model, classes, guidance_scale=3.0):
    """
    CFG sampling for a batch of class labels.
    classes: (N,) tensor of class indices (0-9)
    guidance_scale: w in the CFG equation
    """
    model.eval()
    N   = len(classes)
    x   = torch.randn(N, 1, 28, 28)
    null = torch.full((N,), NULL_CLASS, dtype=torch.long)
    
    for t_idx in reversed(range(T_STEPS)):
        t_batch   = torch.full((N,), t_idx, dtype=torch.long)
        
        # Two forward passes: conditioned and unconditioned
        eps_cond   = model(x, t_batch, classes)
        eps_uncond = model(x, t_batch, null)
        
        # CFG: amplify toward condition direction
        eps_cfg    = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
        
        # DDPM posterior update
        ab_t = alpha_bar[t_idx]
        at   = alphas[t_idx]
        bt   = betas[t_idx]
        x0_pred = ((x - sqrt_1mab[t_idx] * eps_cfg) / sqrt_ab[t_idx]).clamp(-1, 1)
        
        if t_idx > 0:
            ab_prev    = alpha_bar[t_idx - 1]
            bt_tilde   = bt * (1 - ab_prev) / (1 - ab_t)
            mu         = (ab_prev.sqrt() * bt / (1-ab_t)) * x0_pred + \
                         (at.sqrt() * (1-ab_prev) / (1-ab_t)) * x
            x          = mu + bt_tilde.sqrt() * torch.randn_like(x)
        else:
            x = x0_pred
    
    return x


# Generate one of each digit with guidance scale 3.0
print("Generating one of each digit (classes 0–9)...")
classes = torch.arange(10)
generated = cfg_sample(model, classes, guidance_scale=3.0)

fig, axes = plt.subplots(1, 10, figsize=(14, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(generated[i, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"{i}")
    ax.axis("off")
plt.suptitle("CFG guided generation: each column = requested digit", y=1.1)
plt.tight_layout()
plt.show()

## 4 · Guidance Scale Sweep

Show the same digit (class=3) at different guidance scales: $w \in \{0, 1, 3, 5, 10\}$.

In [ ]:
guidance_scales = [0.0, 1.0, 2.0, 4.0, 7.0]
target_class    = 3
n_per_scale     = 4

print("Running guidance scale sweep for digit '3'...")
results = []
for w in guidance_scales:
    classes = torch.full((n_per_scale,), target_class, dtype=torch.long)
    imgs = cfg_sample(model, classes, guidance_scale=w)
    results.append((w, imgs))
    print(f"  w={w:.1f} done")

fig, axes = plt.subplots(len(guidance_scales), n_per_scale,
                          figsize=(n_per_scale * 2, len(guidance_scales) * 2.5))

for row_idx, (w, imgs) in enumerate(results):
    for col_idx in range(n_per_scale):
        axes[row_idx, col_idx].imshow(imgs[col_idx, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
        axes[row_idx, col_idx].axis("off")
    axes[row_idx, 0].set_ylabel(f"w={w:.1f}", rotation=0, labelpad=40, va="center")

plt.suptitle("Guidance scale sweep for digit '3'\n"
             "w=0: unconditioned | w=1: no amplification | w>1: amplified conditioning",
             y=1.02)
plt.tight_layout()
plt.show()

print("\nObservation:")
print("  w=0.0  → unconditioned; may not generate a '3'")
print("  w=1.0  → conditioned but no CFG amplification")
print("  w=3-7  → clearly recognisable '3'; some loss of texture variety")
print("  High w → very sharp but potentially oversaturated / repetitive")

## 5 · Summary — PixelSmith v3.5

```
┌──────────────────────────────────────────────────────────────────────────────┐
│ PixelSmith v3.5 — Conditional DDPM with CFG                                   │
│                                                                                │
│  TRAINING CHANGE: add class embedding; randomly drop it (P_UNCOND=10%)       │
│                                                                                │
│  INFERENCE CHANGE: run U-Net twice per step                                   │
│    ε_cond   = model(x_t, t, class)                                           │
│    ε_uncond = model(x_t, t, null)                                            │
│    ε_cfg    = ε_uncond + w × (ε_cond - ε_uncond)                            │
│                                                                                │
│  COST: 2× U-Net calls per step (+ 2000 passes for T=1000)                   │
│  BENEFIT: controlled generation of specific digit classes                    │
│                                                                                │
│  In Stable Diffusion: replace class embedding with CLIP text encoder output  │
│  and inject via cross-attention instead of time embedding addition           │
└──────────────────────────────────────────────────────────────────────────────┘
```

**Next:** [Schedulers.md](../Schedulers/Schedulers.md) — reduce the 2000 U-Net calls to 40 without retraining.